* list down all the folders

In [ ]:
from pathlib import Path

base_path = Path(r"D:\Dermerzel\SomnasNest\Alzheimer\Data")

folders = [p.name for p in base_path.iterdir() if p.is_dir()]

for folder in folders:
    print(folder)


* load sub-01 EEG data

In [ ]:
pip install mne

In [ ]:
import mne
from pathlib import Path

# Path to subject folder
sub_path = Path(r"D:\Dermerzel\SomnasNest\Alzheimer\Data\sub-01")

# Find the .vhdr file
vhdr_file = list(sub_path.glob("*.vhdr"))[0]

# Load EEG data
raw = mne.io.read_raw_brainvision(vhdr_file, preload=True)

# Print basic info
print(raw)


In [ ]:
# Show channel names
print(raw.ch_names)

* save ngeative class subjects as numpy array

In [ ]:
import mne
from pathlib import Path

print("=== Checking EEG shapes from sub-01 to sub-80 ===")

base_path = Path(r"D:\Dermerzel\SomnasNest\Alzheimer\Data")

lengths = {}
channel_counts = {}

for i in range(1, 81):
    sub_id = f"sub-{i:02d}"
    sub_path = base_path / sub_id

    print("\n----------------------------------")
    print(f"Processing {sub_id}")

    if not sub_path.exists():
        print("❌ Folder not found")
        continue

    vhdr_files = list(sub_path.glob("*.vhdr"))
    if len(vhdr_files) == 0:
        print("⚠️ No .vhdr file found")
        continue

    vhdr_file = vhdr_files[0]
    print(f"Using file: {vhdr_file.name}")

    try:
        raw = mne.io.read_raw_brainvision(vhdr_file, preload=False, verbose=False)
        raw.pick_types(eeg=True)

        n_channels, n_times = raw.get_data(reject_by_annotation="omit").shape

        lengths[sub_id] = n_times
        channel_counts[sub_id] = n_channels

        print(f"✔ Shape: (channels={n_channels}, time={n_times})")

    except Exception as e:
        print(f"❌ Error loading {sub_id}: {e}")

print("\n==================================")
print("=== Summary ===")

if lengths:
    max_length = max(lengths.values())
    min_length = min(lengths.values())
    unique_channels = set(channel_counts.values())

    print(f"Subjects successfully loaded: {len(lengths)}")
    print(f"Maximum time length: {max_length}")
    print(f"Minimum time length: {min_length}")
    print(f"Unique channel counts: {unique_channels}")

    print("\nSubjects with maximum length:")
    for sub, length in lengths.items():
        if length == max_length:
            print(f"  {sub} → {length}")

else:
    print("❌ No valid EEG data found")


In [ ]:
import mne
import numpy as np
from pathlib import Path

print("=== EEG Loading Started ===")

# Base directory
base_path = Path(r"D:\Dermerzel\SomnasNest\Alzheimer\Data")
print(f"Base path set to: {base_path}")

TARGET_LENGTH = 150_000  # <-- FIXED NUMBER OF TIME POINTS

eeg_data_list = []
loaded_subjects = []

# --------------------------------------------------
# STEP 1: Load and crop each subject
# --------------------------------------------------
for i in range(1, 32):
    sub_id = f"sub-{i:02d}"
    sub_path = base_path / sub_id

    print("\n----------------------------------")
    print(f"Processing {sub_id}")
    print(f"Looking in: {sub_path}")

    if not sub_path.exists():
        print(f"❌ Folder not found: {sub_path}")
        continue

    vhdr_files = list(sub_path.glob("*.vhdr"))
    print(f"Found {len(vhdr_files)} .vhdr file(s)")

    if len(vhdr_files) == 0:
        print(f"⚠️ No .vhdr file found in {sub_id}, skipping.")
        continue

    vhdr_file = vhdr_files[0]
    print(f"Using file: {vhdr_file.name}")

    try:
        print("→ Loading EEG data...")
        raw = mne.io.read_raw_brainvision(vhdr_file, preload=True, verbose=False)

        print("→ Picking EEG channels only...")
        raw.pick_types(eeg=True)

        print("→ Extracting NumPy array...")
        data = raw.get_data()

        n_channels, n_times = data.shape
        print(f"✔ Original shape: {data.shape}")

        if n_times < TARGET_LENGTH:
            print(f"⚠️ Skipping {sub_id}: only {n_times} samples (< {TARGET_LENGTH})")
            continue

        # KEEP FIRST 150,000 SAMPLES ONLY
        cropped = data[:, :TARGET_LENGTH].astype(np.float16)

        print(f"✔ Cropped shape: {cropped.shape}")

        eeg_data_list.append(cropped)
        loaded_subjects.append(sub_id)

        print(f"✔ {sub_id} successfully processed.")

    except Exception as e:
        print(f"❌ Error loading {sub_id}: {e}")
        continue

print("\n==================================")
print("EEG Loading Finished")
print(f"Total subjects loaded: {len(eeg_data_list)}")
print(f"Subjects: {loaded_subjects}")

# --------------------------------------------------
# STEP 2: Stack all subjects
# --------------------------------------------------
print("\n→ Stacking all subjects into one NumPy array...")

try:
    ad_negative = np.stack(eeg_data_list, axis=0)
    print("✔ Stacking successful.")
    print("Final shape of ad_negative:", ad_negative.shape)
    print("Data type:", ad_negative.dtype)
except MemoryError as e:
    print("❌ Memory error during stacking!")
    print(e)


In [ ]:
import numpy as np

save_path = r"D:\Dermerzel\SomnasNest\Alzheimer\Data\ad_negative.npy"

np.save(save_path, ad_negative)

print(f"✔ ad_negative saved successfully to:\n{save_path}")
print("Saved shape:", ad_negative.shape)
print("Saved dtype:", ad_negative.dtype)


* saave the positive class as numpy array

In [ ]:
import mne
import numpy as np
from pathlib import Path

print("=== EEG Loading Started (AD POSITIVE) ===")

# Base directory
base_path = Path(r"D:\Dermerzel\SomnasNest\Alzheimer\Data")
print(f"Base path set to: {base_path}")

TARGET_LENGTH = 150_000

eeg_data_list = []
loaded_subjects = []

# --------------------------------------------------
# STEP 1: Load and crop each subject
# --------------------------------------------------
for i in range(32, 81):
    sub_id = f"sub-{i:02d}"
    sub_path = base_path / sub_id

    print("\n----------------------------------")
    print(f"Processing {sub_id}")
    print(f"Looking in: {sub_path}")

    if not sub_path.exists():
        print(f"❌ Folder not found: {sub_path}")
        continue

    vhdr_files = list(sub_path.glob("*.vhdr"))
    print(f"Found {len(vhdr_files)} .vhdr file(s)")

    if len(vhdr_files) == 0:
        print(f"⚠️ No .vhdr file found in {sub_id}, skipping.")
        continue

    vhdr_file = vhdr_files[0]
    print(f"Using file: {vhdr_file.name}")

    try:
        print("→ Loading EEG data...")
        raw = mne.io.read_raw_brainvision(vhdr_file, preload=True, verbose=False)

        print("→ Picking EEG channels only...")
        raw.pick_types(eeg=True)

        print("→ Extracting NumPy array...")
        data = raw.get_data()

        n_channels, n_times = data.shape
        print(f"✔ Original shape: {data.shape}")

        if n_times < TARGET_LENGTH:
            print(f"⚠️ Skipping {sub_id}: only {n_times} samples (< {TARGET_LENGTH})")
            continue

        # KEEP FIRST 150,000 SAMPLES
        cropped = data[:, :TARGET_LENGTH].astype(np.float16)

        print(f"✔ Cropped shape: {cropped.shape}")

        eeg_data_list.append(cropped)
        loaded_subjects.append(sub_id)

        print(f"✔ {sub_id} successfully processed.")

    except Exception as e:
        print(f"❌ Error loading {sub_id}: {e}")
        continue

print("\n==================================")
print("EEG Loading Finished (AD POSITIVE)")
print(f"Total subjects loaded: {len(eeg_data_list)}")
print(f"Subjects: {loaded_subjects}")

# --------------------------------------------------
# STEP 2: Stack all subjects
# --------------------------------------------------
print("\n→ Stacking all subjects into one NumPy array...")

try:
    ad_positive = np.stack(eeg_data_list, axis=0)
    print("✔ Stacking successful.")
    print("Final shape of ad_positive:", ad_positive.shape)
    print("Data type:", ad_positive.dtype)
except MemoryError as e:
    print("❌ Memory error during stacking!")
    print(e)

# --------------------------------------------------
# STEP 3: Save to disk
# --------------------------------------------------
save_path = base_path / "ad_positive.npy"

np.save(save_path, ad_positive)

print(f"\n✔ ad_positive saved successfully to:")
print(save_path)


In [ ]:
import mne
from pathlib import Path

print("=== Checking EEG sampling rates from sub-01 to sub-80 ===")

base_path = Path(r"D:\Dermerzel\SomnasNest\Alzheimer\Data")

sampling_rates = {}

for i in range(1, 81):
    sub_id = f"sub-{i:02d}"
    sub_path = base_path / sub_id

    print("\n----------------------------------")
    print(f"Processing {sub_id}")

    if not sub_path.exists():
        print("❌ Folder not found")
        continue

    vhdr_files = list(sub_path.glob("*.vhdr"))
    if len(vhdr_files) == 0:
        print("⚠️ No .vhdr file found")
        continue

    vhdr_file = vhdr_files[0]
    print(f"Using file: {vhdr_file.name}")

    try:
        raw = mne.io.read_raw_brainvision(vhdr_file, preload=False, verbose=False)
        raw.pick_types(eeg=True)

        sfreq = raw.info["sfreq"]
        sampling_rates[sub_id] = sfreq

        print(f"✔ Sampling rate: {sfreq} Hz")

    except Exception as e:
        print(f"❌ Error loading {sub_id}: {e}")

print("\n==================================")
print("=== Sampling Rate Summary ===")

if sampling_rates:
    unique_sfreqs = sorted(set(sampling_rates.values()))

    print(f"Subjects successfully loaded: {len(sampling_rates)}")
    print(f"Unique sampling rates (Hz): {unique_sfreqs}")

    print("\nPer-subject sampling rates:")
    for sub, sfreq in sampling_rates.items():
        print(f"  {sub} → {sfreq} Hz")
else:
    print("❌ No valid EEG data found")
